# Tầng 0: smoke test cho việc tune CONSTRUCT (Kaggle **GPU T4 ×2**, Omni-3B). Chạy **1 lần**, khoảng 1,5–2,5 giờ
Kế hoạch: `ideas/F1_tuning_plan_kaggle3b_2026-11-28.md`. Chỉ dùng **MER2025-FG** (tập dev). Notebook không giải nén và không đọc OV-MERD+.

Notebook kiểm tra:
1. Môi trường: Python ≥ 3.11, torch ≥ 2.6, transformers 5.24.0, qwen-omni-utils 0.0.9 (in ra backend đọc video).
2. Omni-3B **fp16** trên T4 với cả 4 cấu hình lấy mẫu, trên cùng 12 clip:
   - S0: T = 1,0, top_p = 0,95, top_k = 50 (đúng như P17); 32 mẫu + greedy;
   - S1: T = 0,7;
   - S2: min-p 0,05;
   - S3: ε = 3e-4;
   - S1–S3: 16 mẫu.
   Đo giây/clip, số token, tỉ lệ chạy hết 512 token, và kiểm tra stop string có chặn được lượt hội thoại bịa không.
3. Extractor chính thức Qwen2.5-7B-Instruct **fp16**: thử vLLM 0.34 chia trên 2 GPU (môi trường riêng); không được thì dùng transformers. Đo số prompt/giây.
4. Thống kê nhãn không dùng GT: tỉ lệ mẫu rỗng, mẫu chỉ có từ ngoài từ vựng, số nhóm mỗi mẫu, độ khớp giữa các mẫu.

**Cài đặt:**
- Accelerator **GPU T4 ×2**, Internet **On**.
- Secret **HF_TOKEN**; thêm **MER2026_ZIP_PW** nếu notebook báo zip bị mã hóa.

**Gửi lại cho Claude:** file `/kaggle/working/t0_out.tar` (mô tả, nhãn, log, `t0_report.md`; không có media).

In [ ]:
KOUT, WORK = '/kaggle/working', '/tmp/t1'
import os, sys, time, json, glob, subprocess
T0 = time.time()
os.makedirs(WORK, exist_ok=True)
subprocess.run(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme', shell=True, check=True)
COMMIT = subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout.strip()
print('code commit:', COMMIT)
OV = f'{WORK}/icme/explore/ovmer'; sys.path.insert(0, OV)
import kaggle_t1 as K
K.secrets()
ENV = K.setup(WORK)

open(f'{KOUT}/t0_report.md', 'w').write(f'# Tier 0 smoke test\n\ncode commit {COMMIT}\n\n{ENV}\n\n')

In [ ]:
# data (MER2025-FG only) and both models
ITEMS = K.prepare(WORK, models=(K.OMNI, K.EXTRACTOR))
sh = K.sh; sh(f'df -h {WORK} | tail -1')

In [ ]:
# describe: the same first 12 MER2025-FG clips under each configuration; GPU0 runs S0 then S2, GPU1 runs S1 then S3
N_SMOKE = 12
JOBS = {0: [f'S0:0/1:{N_SMOKE}', f'S2:0/1:{N_SMOKE}'], 1: [f'S1:0/1:{N_SMOKE}', f'S3:0/1:{N_SMOKE}']}
K.run_queues(JOBS, ITEMS, KOUT, budget_h=3.0, hard_h=3.5, t0=time.time(), tar_name='t0_out.tar')
DESCS = sorted(glob.glob(f'{KOUT}/desc_S*_0of1.jsonl')); print(DESCS)

In [ ]:
# label-free description statistics and the projection for 1,000 clips on 2 GPUs
subprocess.run([sys.executable, f'{OV}/t1_inspect.py', '--descs', ','.join(DESCS), '--n_clips', '1000', '--gpus', '2',
                '--report', f'{KOUT}/t0_report.md'], check=True)
bad = [json.loads(l) for d in DESCS for l in open(d) if not json.loads(l).get('ok')]
print('failed clips:', len(bad), [b['err'][:120] for b in bad[:3]])

In [ ]:
# extractor: vLLM 0.34 (own environment, tensor parallel over the 2 T4s); fall back to transformers if vLLM does not run
sh('pip -q install uv', check=False)
VPY = f'{WORK}/vv/bin/python'
sh(f'uv venv -q -p {sys.executable} {WORK}/vv && uv pip install -q -p {VPY} "vllm==0.34.0"', check=False)
EX = f'{OV}/t1_extract.py'; LOGX = f'{KOUT}/log_extract.txt'
def extract(desc, out, mode, engine, py, env_extra=None):
    cmd = [py, EX, '--descs', desc, '--out', out, '--engine', engine, '--mode', mode, '--tp', '2']
    env = dict(os.environ, **(env_extra or {}))
    with open(LOGX, 'a') as f:
        f.write(f'\n== {engine} {mode} {os.path.basename(desc)}\n'); f.flush()
        t = time.time(); r = subprocess.run(cmd, env=env, stdout=f, stderr=subprocess.STDOUT)
    print(f'{engine} {mode} {os.path.basename(desc)}: rc {r.returncode}, {time.time() - t:.0f} s', flush=True)
    return r.returncode == 0
S0 = f'{KOUT}/desc_S0_0of1.jsonl'
ENGINE, PY, XENV = 'vllm', VPY, None
if not (os.path.exists(VPY) and extract(S0, f'{KOUT}/labels_S0_X0.jsonl', 'official', 'vllm', VPY)):
    ENGINE, XENV = 'vllm', {'NCCL_P2P_DISABLE': '1'}
    if os.path.exists(f'{KOUT}/labels_S0_X0.jsonl'): os.remove(f'{KOUT}/labels_S0_X0.jsonl')
    if not (os.path.exists(VPY) and extract(S0, f'{KOUT}/labels_S0_X0.jsonl', 'official', 'vllm', VPY, XENV)):
        ENGINE, PY, XENV = 'hf', sys.executable, None
        if os.path.exists(f'{KOUT}/labels_S0_X0.jsonl'): os.remove(f'{KOUT}/labels_S0_X0.jsonl')
        assert extract(S0, f'{KOUT}/labels_S0_X0.jsonl', 'official', 'hf', PY), 'both extractor engines failed: see log_extract.txt'
print('extractor engine:', ENGINE, XENV or '')
for cfg in ('S1', 'S2', 'S3'):
    extract(f'{KOUT}/desc_{cfg}_0of1.jsonl', f'{KOUT}/labels_{cfg}_X0.jsonl', 'official', ENGINE, PY, XENV)
extract(S0, f'{KOUT}/labels_S0_X1.jsonl', 'greedy', ENGINE, PY, XENV)
extract(S0, f'{KOUT}/labels_S0_X2.jsonl', 'k', ENGINE, PY, XENV)
speed = [l.strip() for l in open(LOGX) if l.startswith('==') or 'prompts/s' in l]
print('\n'.join(speed[-20:]))
open(f'{KOUT}/t0_report.md', 'a').write(f'## Extractor\n\nengine {ENGINE} {XENV or ""}\n\n```\n' + '\n'.join(speed) + '\n```\n\n')

In [ ]:
# label-free label statistics, report, archive
LABS = sorted(glob.glob(f'{KOUT}/labels_*.jsonl'))
subprocess.run([sys.executable, f'{OV}/t1_inspect.py', '--labels', ','.join(LABS), '--mapping',
                f'{WORK}/affectgpt/AffectGPT-RL/emotion_wheel/wheel_mapping.npz', '--report', f'{KOUT}/t0_report.md'], check=True)
open(f'{KOUT}/t0_report.md', 'a').write(f'total {(time.time() - T0) / 3600:.2f} h\n')
K.snapshot(KOUT, 't0_out.tar'); print(open(f'{KOUT}/t0_report.md').read())